# Unseen-household generalization

This notebook reports a deliberate household holdout experiment. The fixed household assignment from Notebook 04B defines the intentional holdout; Notebook 05 and this notebook exclude those households from model selection, fitting, calibration, and threshold selection. The held-out households' historical smart-meter observations may still contribute to already-engineered lag/profile predictors, while their labels do not inform the fitted model or decision threshold.

The principal comparison is future performance for the known-household development pool versus the intentional holdout. This estimates transfer to deliberately unseen households that have enough historical meter data to construct predictors.

The natural unseen cohort is a separate, late-entry group with 16 households and 277 test rows. It is reported descriptively as exploratory because support is small. Keep this cohort distinct from the intentional holdout; they answer different questions. No models, thresholds, or hyperparameters are changed based on either test result.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, brier_score_loss,
                             confusion_matrix, f1_score, log_loss, precision_score, recall_score,
                             roc_auc_score, accuracy_score)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

ROOT=Path.cwd().resolve()
while not (ROOT/"data"/"processed").exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/"data"/"processed").exists(): raise RuntimeError("Could not locate project root")
DATA_DIR=ROOT/"data"/"processed"
TABLE_DIR=ROOT/"reports"/"tables"/"10_unseen_household_generalization"
FIGURE_DIR=ROOT/"reports"/"figures"/"10_unseen_household_generalization"
MODEL_DIR=ROOT/"models"/"10_unseen_household_generalization"
for d in (TABLE_DIR,FIGURE_DIR,MODEL_DIR): d.mkdir(parents=True,exist_ok=True)
MASTER_PATH=DATA_DIR/"04_master_split.csv"
ASSIGN_PATH=DATA_DIR/"04b_household_holdout_assignment.csv"
HOLDOUT_MANIFEST_PATH=ROOT/"reports"/"tables"/"04b_unseen_household_holdout_design"/"04b_unseen_household_holdout_manifest.json"
SELECTED_PATH=ROOT/"reports"/"tables"/"05_model_training_tuning"/"05_selected_models.json"
SELECTION_MANIFEST_PATH=ROOT/"reports"/"tables"/"05_model_training_tuning"/"05_model_training_tuning_manifest.json"
DATA_PATHS={"P95":DATA_DIR/"05_model_ready_m0_p95.csv","P90":DATA_DIR/"05_model_ready_m0_p90.csv"}
INTERNAL_STAGES=("model_development","calibration","threshold_selection")
N_BOOTSTRAP=1000; BOOTSTRAP_SEED=20261008; THRESHOLD_GRID=np.round(np.arange(0.01,1.00,0.01),2)

def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))
def sha256(path):
 h=hashlib.sha256()
 with Path(path).open("rb") as f:
  for block in iter(lambda:f.read(8*1024*1024),b""): h.update(block)
 return h.hexdigest()
def numeric_x(frame,features): return frame[features].apply(lambda c:pd.to_numeric(c,errors="coerce").astype("float64"))
def probability_logit(p):
 p=np.clip(np.asarray(p,dtype=float),1e-6,1-1e-6)
 return np.log(p/(1-p)).reshape(-1,1)


## 1. Verify frozen household assignment and select only permitted labelled rows

The assignment file is joined to the master split by household. Source CSV row IDs are scanned first; feature and target columns are then parsed only for non-held-out internal rows and reserved outer-test rows. No labelled historical rows belonging to the intentional holdout are loaded.

In [ ]:
for path in [MASTER_PATH,ASSIGN_PATH,HOLDOUT_MANIFEST_PATH,SELECTED_PATH,SELECTION_MANIFEST_PATH,*DATA_PATHS.values()]:
 if not path.exists(): raise FileNotFoundError(f"Missing prerequisite: {path}")
master=pd.read_csv(MASTER_PATH,dtype={"row_id":"string","household_id":"string"})
master["target_date"]=pd.to_datetime(master.target_date,errors="raise")
assignment=pd.read_csv(ASSIGN_PATH,dtype={"household_id":"string","holdout_group":"string"})
assignment_manifest=read_json(HOLDOUT_MANIFEST_PATH); selected=read_json(SELECTED_PATH); selection_manifest=read_json(SELECTION_MANIFEST_PATH)
if not selection_manifest.get("intentional_holdout_households_excluded_from_model_selection",False): raise ValueError("Notebook 05 did not exclude intentional holdout households from model selection")
if selection_manifest.get("intentional_holdout_assignment_sha256") != sha256(ASSIGN_PATH): raise ValueError("Notebook 05 selection used a different 04B holdout assignment")
if not assignment_manifest.get("validation",{}).get("assignment_groups_are_disjoint_and_complete",False): raise ValueError("Notebook 04B assignment validation did not pass")
if assignment_manifest.get("future_target_outcomes_used_for_assignment",True): raise ValueError("Holdout assignment used future outcomes")
if master.row_id.duplicated().any() or assignment.household_id.duplicated().any(): raise ValueError("Master row IDs and household assignments must be unique")
if set(master.household_id.astype(str))!=set(assignment.household_id.astype(str)): raise ValueError("04B assignment does not cover the frozen master households exactly")
if set(selected)!={"P95","P90"}: raise ValueError("Notebook 05 selected-model handoff missing a target")
intentional_ids=set(assignment.loc[assignment.holdout_group.eq("intentional_holdout"),"household_id"].astype(str))
if len(intentional_ids)!=int(assignment_manifest["holdout_group_values"]["intentional_holdout"]): raise ValueError("Intentional holdout household count differs from the frozen manifest")

master=master.merge(assignment,on="household_id",how="left",validate="many_to_one")
if master.holdout_group.isna().any(): raise ValueError("Some master rows lack a frozen holdout group")
test_groups=set(master.loc[master.outer_split.eq("test"),"holdout_group"].unique())
if not {"development_pool","intentional_holdout","natural_unseen_test"}.issubset(test_groups): raise ValueError(f"Expected test groups not found: {test_groups}")

# Internal labels are authorized only for non-held-out households. Test IDs are included for final cohort scoring.
permitted_internal=set(master.loc[master.internal_split.isin(INTERNAL_STAGES)&~master.household_id.astype(str).isin(intentional_ids),"row_id"].astype(str))
outer_test_ids=set(master.loc[master.outer_split.eq("test"),"row_id"].astype(str))
load_ids=permitted_internal|outer_test_ids

def read_permitted_rows(path,ids):
 ids=set(map(str,ids)); rowids=pd.read_csv(path,usecols=["row_id"],dtype={"row_id":"string"}).row_id
 positions=np.flatnonzero(rowids.isin(ids).to_numpy()); found=set(rowids.iloc[positions].astype(str))
 if found!=ids: raise ValueError(f"{path.name}: master/data row mismatch missing={len(ids-found)}")
 keep=set(positions.tolist()); frame=pd.read_csv(path,dtype={"row_id":"string","household_id":"string"},skiprows=lambda line:line>0 and (line-1) not in keep)
 if set(frame.row_id.astype(str))!=ids or frame.row_id.duplicated().any(): raise ValueError(f"{path.name}: selective loader returned invalid rows")
 frame["measurement_date"]=pd.to_datetime(frame.measurement_date,errors="raise")
 meta=master[["row_id","outer_split","internal_split","test_cohort","target_date","holdout_group"]]
 frame=frame.merge(meta,on="row_id",how="left",validate="one_to_one")
 if frame.outer_split.isna().any() or not frame.measurement_date.eq(frame.target_date).all(): raise ValueError(f"{path.name}: unmatched split/date metadata")
 if frame.loc[frame.outer_split.eq("development")&frame.household_id.astype(str).isin(intentional_ids)].shape[0]: raise ValueError("Held-out historical target rows were loaded")
 return frame

frames={target:read_permitted_rows(path,load_ids) for target,path in DATA_PATHS.items()}
for target,frame in frames.items():
 fit=frame.loc[frame.outer_split.eq("development")&frame.internal_split.eq("model_development")]
 cal=frame.loc[frame.outer_split.eq("development")&frame.internal_split.eq("calibration")]
 th=frame.loc[frame.outer_split.eq("development")&frame.internal_split.eq("threshold_selection")]
 if any(set(x.household_id.astype(str))&intentional_ids for x in (fit,cal,th)): raise ValueError(f"{target}: holdout households appear in an internal labelled block")
 if any(x.empty for x in (fit,cal,th)): raise ValueError(f"{target}: empty fit/calibration/threshold block after exclusion")

cohort_map={"development_pool":"known_household_future_test","intentional_holdout":"intentional_heldout_future_test","natural_unseen_test":"natural_unseen_exploratory"}
cohort_records=[]
for target,frame in frames.items():
 for group,label in cohort_map.items():
  part=frame.loc[frame.outer_split.eq("test")&frame.holdout_group.eq(group)]
  cohort_records.append({"target":target,"cohort":label,"holdout_group":group,"rows":len(part),"households":int(part.household_id.nunique()),"events":int(part.target.sum()),"prevalence":float(part.target.mean()) if len(part) else np.nan,
                         "start_date":part.measurement_date.min(),"end_date":part.measurement_date.max()})
cohort_summary=pd.DataFrame(cohort_records); cohort_summary.to_csv(TABLE_DIR/"10_test_cohort_summary.csv",index=False); display(cohort_summary)


## 2. Refit the frozen models without holdout labels, calibrate, and choose thresholds

The family and parameters are read from Notebook 05. Preprocessing and classifier fitting use non-held-out `model_development`; Platt calibration uses non-held-out `calibration`; the fixed F1 rule selects thresholds on non-held-out `threshold_selection`. These settings are not revised based on household-holdout or test results.

In [ ]:
def build_estimator(family,params,yfit):
 params=dict(params)
 if family=="XGBoost":
  mode=params.pop("scale_pos_weight_mode","one"); ratio=float((yfit==0).sum()/max((yfit==1).sum(),1))
  scales={"one":1.0,"sqrt_ratio":ratio**0.5,"half_ratio":ratio/2.0,"full_ratio":ratio}
  if mode not in scales: raise ValueError(f"Unknown scale_pos_weight_mode={mode}")
  model=XGBClassifier(**params,scale_pos_weight=scales[mode],eval_metric="logloss",tree_method="hist",n_jobs=2,random_state=42,verbosity=0)
  return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",model)])
 if family=="Random Forest":
  return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",RandomForestClassifier(**params,n_jobs=2,random_state=42))])
 if family=="Logistic Regression":
  return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",LogisticRegression(**params,solver="lbfgs",max_iter=2000,random_state=42))])
 if family=="CatBoost":
  return CatBoostClassifier(**params,loss_function="Logloss",random_seed=42,verbose=False,allow_writing_files=False,thread_count=2)
 raise ValueError(f"Unsupported frozen family: {family}")

def choose_threshold(y,p,target):
 rows=[]
 for th in THRESHOLD_GRID:
  pred=(p>=th).astype("int8"); rows.append({"threshold":float(th),"precision":precision_score(y,pred,zero_division=0),"recall":recall_score(y,pred,zero_division=0),"f1":f1_score(y,pred,zero_division=0)})
 curve=pd.DataFrame(rows); best=curve.sort_values(["f1","precision","threshold"],ascending=[False,False,False],kind="stable").iloc[0]
 return curve,best

models={}; calibrators={}; thresholds={}; test_probabilities={}; model_bundles={}; training_rows=[]; threshold_rows=[]; curve_rows=[]
for target,frame in frames.items():
 spec=selected[target]; features=spec["feature_list"]; family=spec["model_family"]
 fit=frame.loc[frame.outer_split.eq("development")&frame.internal_split.eq("model_development")].copy()
 cal=frame.loc[frame.outer_split.eq("development")&frame.internal_split.eq("calibration")].copy()
 th=frame.loc[frame.outer_split.eq("development")&frame.internal_split.eq("threshold_selection")].copy()
 yfit=fit.target.astype("int8").to_numpy(); ycal=cal.target.astype("int8").to_numpy()
 estimator=build_estimator(family,spec["parameters"],yfit); estimator.fit(numeric_x(fit,features),yfit)
 pcal=estimator.predict_proba(numeric_x(cal,features))[:,1]
 calibrator=LogisticRegression(C=1.0,solver="lbfgs",max_iter=2000,random_state=42).fit(probability_logit(pcal),ycal)
 pth_raw=estimator.predict_proba(numeric_x(th,features))[:,1]; pth=calibrator.predict_proba(probability_logit(pth_raw))[:,1]
 curve,best=choose_threshold(th.target.astype("int8").to_numpy(),pth,target)
 threshold=float(best.threshold); models[target]=estimator; calibrators[target]=calibrator; thresholds[target]=threshold
 curve.insert(0,"target",target); curve_rows.append(curve)
 threshold_rows.append({"target":target,"model_family":family,"threshold":threshold,"fit_rows":len(fit),"fit_households":fit.household_id.nunique(),
                       "calibration_rows":len(cal),"threshold_rows":len(th),"threshold_events":int(th.target.sum()),"selection_policy":"maximize F1; ties by precision then higher threshold"})
 training_rows.append({"target":target,"model_family":family,"parameters":json.dumps(spec["parameters"],sort_keys=True),"fit_rows":len(fit),"fit_events":int(yfit.sum()),
                       "fit_households":int(fit.household_id.nunique()),"intentional_holdout_ids_in_fit":0,"calibration_rows":len(cal),"threshold_selection_rows":len(th)})
 test=frame.loc[frame.outer_split.eq("test")].copy(); raw=estimator.predict_proba(numeric_x(test,features))[:,1]
 test_probabilities[target]=calibrator.predict_proba(probability_logit(raw))[:,1]
 model_bundles[target]={"target":target,"model_family":family,"selected_parameters":spec["parameters"],"features":features,"base_estimator":estimator,
                        "platt_calibrator":calibrator,"threshold":threshold,"fit_split":"model_development_nonholdout",
                        "calibration_split":"calibration_nonholdout","threshold_split":"threshold_selection_nonholdout",
                        "intentional_holdout_households_excluded_from_fit_calibration_threshold":True,
                        "notebook05_selection_included_holdout_households":False}
 print(f"{target}: {family}; fit rows={len(fit):,}; fit households={fit.household_id.nunique():,}; held-out test rows={(test.holdout_group=='intentional_holdout').sum():,}; threshold={threshold:.2f}")

threshold_table=pd.DataFrame(threshold_rows); fit_summary=pd.DataFrame(training_rows); threshold_curves=pd.concat(curve_rows,ignore_index=True)
threshold_table.to_csv(TABLE_DIR/"10_nonholdout_thresholds.csv",index=False)
fit_summary.to_csv(TABLE_DIR/"10_fit_summary.csv",index=False)
threshold_curves.to_csv(TABLE_DIR/"10_threshold_selection_curves.csv",index=False)
display(threshold_table); display(fit_summary)


## 3. Evaluate known, intentional-holdout, and natural-unseen cohorts

development_pool measures future prediction for households represented during model development. intentional_holdout measures transfer to the fixed 20% household sample withheld from all labelled selection and fitting stages. natural_unseen_test represents late-entry households and is a separate, small exploratory cohort (16 households, 277 rows). Do not combine the two unseen cohorts or treat the natural cohort as confirmatory.

In [ ]:
def score_metrics(y,p,threshold,weights=None):
 y=np.asarray(y,dtype="int8"); pred=(np.asarray(p)>=threshold).astype("int8")
 tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1],sample_weight=weights).ravel()
 return {"pr_auc":average_precision_score(y,p,sample_weight=weights),"roc_auc":roc_auc_score(y,p,sample_weight=weights),
         "brier_score":brier_score_loss(y,p,sample_weight=weights),"log_loss":log_loss(y,p,labels=[0,1],sample_weight=weights),
         "precision":precision_score(y,pred,zero_division=0,sample_weight=weights),"recall":recall_score(y,pred,zero_division=0,sample_weight=weights),
         "f1":f1_score(y,pred,zero_division=0,sample_weight=weights),"balanced_accuracy":balanced_accuracy_score(y,pred,sample_weight=weights),
         "accuracy":accuracy_score(y,pred,sample_weight=weights),"tn":tn,"fp":fp,"fn":fn,"tp":tp}

score_rows=[]; prediction_frames=[]
for target,frame in frames.items():
 mask=frame.outer_split.eq("test").to_numpy(); part=frame.loc[mask].copy(); probability=test_probabilities[target]
 if len(probability)!=len(part): raise ValueError(f"{target}: test prediction count mismatch")
 part["calibrated_probability"]=probability; part["locked_threshold"]=thresholds[target]; part["predicted_label"]=(probability>=thresholds[target]).astype("int8")
 prediction_frames.append(part[["row_id","household_id","measurement_date","target","holdout_group","test_cohort","calibrated_probability","locked_threshold","predicted_label"]].assign(target_name=target))
 for group,label in (("development_pool","known_household_future_test"),("intentional_holdout","intentional_heldout_future_test"),("natural_unseen_test","natural_unseen_exploratory")):
  use=part.holdout_group.eq(group).to_numpy()
  if not use.any(): continue
  y=part.loc[use,"target"].astype("int8").to_numpy(); p=probability[use]
  if np.unique(y).size<2: raise ValueError(f"{target} {label}: metrics require both target classes")
  values=score_metrics(y,p,thresholds[target])
  score_rows.append({"target":target,"cohort":label,"holdout_group":group,"rows":int(use.sum()),"households":int(part.loc[use,"household_id"].nunique()),
                     "events":int(y.sum()),"prevalence":float(y.mean()),"threshold":thresholds[target],**values})

final_metrics=pd.DataFrame(score_rows); final_metrics.to_csv(TABLE_DIR/"10_unseen_household_metrics.csv",index=False)
final_predictions=pd.concat(prediction_frames,ignore_index=True); final_predictions.to_csv(TABLE_DIR/"10_row_level_predictions.csv",index=False)
display(final_metrics)


## 4. Household-cluster bootstrap and known-versus-held-out differences

Bootstrap intervals resample household IDs within each cohort. The known and intentional-holdout cohorts are disjoint, so their difference intervals use independent household draws. The small natural-unseen group is reported descriptively without a strong inferential claim.

In [ ]:
rng=np.random.default_rng(BOOTSTRAP_SEED)
BOOT_METRICS=("pr_auc","roc_auc","brier_score","log_loss","precision","recall","f1","balanced_accuracy")
bootstrap_rows=[]; difference_rows=[]
for target,frame in frames.items():
 data=frame.loc[frame.outer_split.eq("test")].copy(); all_prob=test_probabilities[target]
 cohort_data={}
 for group,label in (("development_pool","known_household_future_test"),("intentional_holdout","intentional_heldout_future_test")):
  mask=data.holdout_group.eq(group).to_numpy(); subset=data.loc[mask]; y=subset.target.astype("int8").to_numpy(); p=all_prob[mask]
  hh=subset.household_id.astype(str).to_numpy(); codes,unique=pd.factorize(hh,sort=True)
  if len(unique)<2: raise ValueError(f"{target}: too few households in {label} for bootstrap")
  cohort_data[group]={"label":label,"y":y,"p":p,"codes":codes,"households":unique,"threshold":thresholds[target],"samples":{m:[] for m in BOOT_METRICS}}
 for _ in range(N_BOOTSTRAP):
  for group,cohort in cohort_data.items():
   draw=rng.integers(0,len(cohort["households"]),size=len(cohort["households"]))
   weights=np.bincount(draw,minlength=len(cohort["households"]))[cohort["codes"]].astype(float)
   values=score_metrics(cohort["y"],cohort["p"],cohort["threshold"],weights=weights)
   for metric in BOOT_METRICS: cohort["samples"][metric].append(values[metric])
 for group,cohort in cohort_data.items():
  point=final_metrics.loc[(final_metrics.target==target)&(final_metrics.holdout_group==group)]
  for metric in BOOT_METRICS:
   vals=cohort["samples"][metric]; lo,hi=np.quantile(vals,[.025,.975])
   bootstrap_rows.append({"target":target,"cohort":cohort["label"],"metric":metric,"point_estimate":float(point[metric].iloc[0]),"ci_95_lower":float(lo),"ci_95_upper":float(hi),
                         "bootstrap_replicates":len(vals),"households":len(cohort["households"]),"resampling_unit":"household"})
 known=cohort_data["development_pool"]; hold=cohort_data["intentional_holdout"]
 for metric in BOOT_METRICS:
  delta_samples=np.asarray(hold["samples"][metric])-np.asarray(known["samples"][metric])
  lo,hi=np.quantile(delta_samples,[.025,.975])
  point_known=final_metrics.loc[(final_metrics.target==target)&final_metrics.holdout_group.eq("development_pool"),metric].iloc[0]
  point_hold=final_metrics.loc[(final_metrics.target==target)&final_metrics.holdout_group.eq("intentional_holdout"),metric].iloc[0]
  difference_rows.append({"target":target,"metric":metric,"known_estimate":point_known,"intentional_holdout_estimate":point_hold,"delta_holdout_minus_known":point_hold-point_known,
                          "ci_95_lower":float(lo),"ci_95_upper":float(hi),"bootstrap_replicates":N_BOOTSTRAP,
                          "known_households":len(known["households"]),"holdout_households":len(hold["households"]),"interpretation":"higher favors holdout" if metric not in ("brier_score","log_loss") else "lower favors holdout"})

bootstrap_ci=pd.DataFrame(bootstrap_rows); generalization_diff=pd.DataFrame(difference_rows)
bootstrap_ci.to_csv(TABLE_DIR/"10_household_bootstrap_confidence_intervals.csv",index=False)
generalization_diff.to_csv(TABLE_DIR/"10_known_vs_holdout_differences.csv",index=False)
display(generalization_diff.loc[generalization_diff.metric.isin(["pr_auc","roc_auc","brier_score","recall","f1"])])


## 5. Visualize known-versus-held-out generalization

In [ ]:
plot_metrics=("pr_auc","roc_auc","brier_score","recall","f1")
fig,axes=plt.subplots(1,2,figsize=(13,6))
for ax,target in zip(axes,("P95","P90")):
 part=generalization_diff.loc[generalization_diff.target.eq(target)&generalization_diff.metric.isin(plot_metrics)].copy()
 part["metric"]=pd.Categorical(part.metric,categories=plot_metrics,ordered=True); part=part.sort_values("metric")
 yy=np.arange(len(part)); center=part.delta_holdout_minus_known.to_numpy(); low=center-part.ci_95_lower.to_numpy(); high=part.ci_95_upper.to_numpy()-center
 ax.errorbar(center,yy,xerr=np.vstack([low,high]),fmt="o",capsize=3,label="Intentional holdout − known; 95% CI")
 ax.axvline(0,color="gray",linestyle="--"); ax.set_yticks(yy,part.metric.astype(str)); ax.invert_yaxis()
 ax.set(title=f"{target}: household-transfer difference",xlabel="Intentional holdout minus known future test")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"10_known_vs_intentional_holdout_metric_differences.png",dpi=160,bbox_inches="tight"); plt.show()

# Cohort prevalence/support and point estimates; natural unseen is shown as exploratory without CIs.
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,target in zip(axes,("P95","P90")):
 part=final_metrics.loc[final_metrics.target.eq(target)]
 labels=["known household","intentional holdout","natural unseen\n(exploratory)"]
 groups=["known_household_future_test","intentional_heldout_future_test","natural_unseen_exploratory"]
 x=np.arange(3); values=[part.loc[part.cohort.eq(g),"pr_auc"].iloc[0] for g in groups]
 counts=[part.loc[part.cohort.eq(g),"households"].iloc[0] for g in groups]
 ax.bar(x,values,color=["#376a9e","#db8a32","#999999"]); ax.set_xticks(x,labels); ax.set_ylabel("PR-AUC (average precision)"); ax.set_title(f"{target}: PR-AUC by household cohort")
 for i,(v,h) in enumerate(zip(values,counts)): ax.text(i,v+.004,f"{v:.3f}\n{int(h)} HH",ha="center",va="bottom",fontsize=9)
fig.tight_layout(); fig.savefig(FIGURE_DIR/"10_cohort_pr_auc_comparison.png",dpi=160,bbox_inches="tight"); plt.show()


## 6. Validate the held-out experiment and save its manifest

In [ ]:
for target,bundle in model_bundles.items(): joblib.dump(bundle,MODEL_DIR/f"{target.lower()}_unseen_household.joblib")
roundtrip_details={}; roundtrip_ok=True
for target in model_bundles:
 loaded=joblib.load(MODEL_DIR/f"{target.lower()}_unseen_household.joblib")
 test_sample=frames[target].loc[frames[target].outer_split.eq("test")&frames[target].holdout_group.eq("intentional_holdout")].head(12)
 raw=loaded["base_estimator"].predict_proba(numeric_x(test_sample,loaded["features"]))[:,1]
 calibrated=loaded["platt_calibrator"].predict_proba(probability_logit(raw))[:,1]
 valid=np.isfinite(calibrated).all() and ((calibrated>=0)&(calibrated<=1)).all() and len(calibrated)==len(test_sample)
 roundtrip_ok=roundtrip_ok and bool(valid); roundtrip_details[target]={"rows":len(test_sample),"valid_probabilities":bool(valid)}

checks=[]
def add_check(name,passed,detail): checks.append({"check":name,"passed":bool(passed),"detail":str(detail)})
add_check("04b_assignment_manifest_and_groups_valid",bool(assignment_manifest.get("validation",{}).get("assignment_groups_are_disjoint_and_complete",False)),len(intentional_ids))
add_check("intentional_holdout_absent_from_all_fit_and_policy_stages",all(not(set(frames[t].loc[frames[t].internal_split.eq(stage),"household_id"].astype(str))&intentional_ids) for t in frames for stage in INTERNAL_STAGES),"Fit, calibration, and threshold-selection labels excluded")
add_check("frozen_notebook05_families_and_parameters_used",all(model_bundles[t]["model_family"]==selected[t]["model_family"] and model_bundles[t]["selected_parameters"]==selected[t]["parameters"] for t in frames),{t:selected[t]["model_family"] for t in selected})
add_check("dedicated_holdout_test_support_present",all(final_metrics.loc[(final_metrics.target==t)&final_metrics.holdout_group.eq("intentional_holdout"),"events"].iloc[0]>=100 for t in frames),"P95 and P90 support checked")
add_check("known_and_holdout_cohorts_reported",set(final_metrics.cohort)=={"known_household_future_test","intentional_heldout_future_test","natural_unseen_exploratory"},sorted(final_metrics.cohort.unique()))
add_check("household_bootstrap_intervals_completed",bool(bootstrap_ci.bootstrap_replicates.eq(N_BOOTSTRAP).all() and generalization_diff.bootstrap_replicates.eq(N_BOOTSTRAP).all()),f"{N_BOOTSTRAP} replicates")
add_check("model_bundles_reload_and_predict",roundtrip_ok,roundtrip_details)
add_check("notebook05_selection_excludes_intentional_holdout",bool(selection_manifest.get("intentional_holdout_households_excluded_from_model_selection",False)) and selection_manifest.get("intentional_holdout_assignment_sha256")==sha256(ASSIGN_PATH) and all(not b["notebook05_selection_included_holdout_households"] for b in model_bundles.values()),"Notebook 05 selection manifest confirms exclusion under the same frozen 04B assignment")
validation=pd.DataFrame(checks); display(validation); validation.to_csv(TABLE_DIR/"10_final_validation.csv",index=False)
if not validation.passed.all(): raise AssertionError("Notebook 10 validation failed")
manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"stage":"10_unseen_household_generalization",
 "assignment_sha256":sha256(ASSIGN_PATH),"assignment_manifest_sha256":sha256(HOLDOUT_MANIFEST_PATH),"notebook05_selected_models_sha256":sha256(SELECTED_PATH),
 "intentional_holdout_households":len(intentional_ids),"fit_calibration_threshold_household_exclusion":True,
 "no_retuning_on_holdout_results":True,"selection_independent_of_intentional_holdout":True,"notebook05_selection_manifest_sha256":sha256(SELECTION_MANIFEST_PATH),
 "threshold_policy":"maximize F1 on non-held-out threshold_selection; ties by precision then higher threshold",
 "cohorts":cohort_summary.to_dict(orient="records"),"models":{t:{"family":model_bundles[t]["model_family"],"threshold":thresholds[t],"bundle":str(MODEL_DIR/f"{t.lower()}_unseen_household.joblib")} for t in model_bundles},
 "bootstrap":{"resampling_unit":"household","replicates":N_BOOTSTRAP,"seed":BOOTSTRAP_SEED,"known_vs_holdout":"independent cluster samples; cohorts are disjoint"},
 "natural_unseen_interpretation":"Descriptive/exploratory only; 16 households.","validation_passed":bool(validation.passed.all()),"checks":checks}
(TABLE_DIR/"10_unseen_household_generalization_manifest.json").write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
print("UNSEEN-HOUSEHOLD EXPERIMENT COMPLETE — frozen Notebook 05 settings retained; no tuning on holdout results.")


## Interpretation and handoff

Compare the known-household and intentional-holdout estimates with their household-bootstrap intervals. A substantial fall in held-out performance suggests transfer is harder than future prediction for households represented during fitting. Similar performance supports transfer within the limits of the design.

The natural unseen cohort is supplementary and exploratory because it contains 16 households and 277 rows. It is not the same cohort as the deliberate 20% holdout and should not be pooled with it.

Notebook 05 model selection and Notebook 10 fitting, calibration, and threshold selection all exclude intentional-holdout households. Their labelled future test outcomes are used only for the final generalization evaluation.